In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load your cleaned regression data 
# (Make sure this is the file where the .1 columns are already dropped)
df = pd.read_csv('../data/cleaned_water_data.csv')


# 2. Separate Inputs (Raw Water) and Targets (Treated Water)
features_x = [col for col in df.columns if col.startswith('RW')]
targets_y = [col for col in df.columns if col.startswith('TW')]

X = df[features_x]
Y = df[targets_y]

# 3. Train/Test Split (80% Train, 20% Test)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# 4. SCALE THE INPUTS (Mandatory for Neural Nets, KNN, and Linear Models)
# We ONLY scale the Inputs (X). We leave Targets (Y) in their natural units so our errors make physical sense.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data Prepped and Scaled!")
print(f"Inputs (X): {X_train_scaled.shape[1]} Raw Water parameters")
print(f"Targets (Y): {Y_train.shape[1]} Treated Water parameters to predict")
print(f"Training Rows: {X_train_scaled.shape[0]} | Testing Rows: {X_test_scaled.shape[0]}")

Data Prepped and Scaled!
Inputs (X): 15 Raw Water parameters
Targets (Y): 16 Treated Water parameters to predict
Training Rows: 337 | Testing Rows: 85


In [4]:
import numpy as np
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor

# 1. Import all the mathematical families
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, 
    GradientBoostingRegressor, 
    AdaBoostRegressor,
    HistGradientBoostingRegressor
)
from sklearn.neural_network import MLPRegressor

# Ignore annoying convergence warnings from linear models pushing too hard
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# 2. Initialize the 12 Competitors
# Note: Algorithms that only predict one target natively are wrapped in MultiOutputRegressor
regression_models = {
    # Linear Family
    "Linear (Standard)": LinearRegression(),
    "Linear (Ridge)": Ridge(alpha=1.0, random_state=42),
    "Linear (Lasso)": Lasso(alpha=0.1, random_state=42),
    "Linear (ElasticNet)": ElasticNet(alpha=0.1, random_state=42),
    
    # Distance / Geometric Family
    "Distance (KNN)": KNeighborsRegressor(n_neighbors=5),
    "Distance (SVR)": MultiOutputRegressor(SVR(kernel='rbf', C=1.0)),
    
    # Tree Family
    "Tree (Decision Tree)": DecisionTreeRegressor(random_state=42),
    "Tree (Random Forest)": RandomForestRegressor(n_estimators=100, random_state=42),
    
    # Boosting Family (The heavy hitters for tabular data)
    "Boosting (AdaBoost)": MultiOutputRegressor(AdaBoostRegressor(random_state=42)),
    "Boosting (Gradient Boost)": MultiOutputRegressor(GradientBoostingRegressor(random_state=42)),
    "Boosting (Hist Grad Boost)": MultiOutputRegressor(HistGradientBoostingRegressor(random_state=42)), # Sklearn's ultra-fast XGBoost equivalent
    
    # Neural Network Family
    "Neural Net (MLP)": MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=2000, random_state=42)
}

print("🥊 THE 12-ALGORITHM MEGA BAKE-OFF 🥊\n" + "="*90)
print(f"{'Algorithm Name':<30} | {'Avg R² Score (Higher=Better)':<30} | {'Avg RMSE (Lower=Better)'}")
print("-" * 90)

regression_results = {}

# 3. The Training Loop
for name, model in regression_models.items():
    try:
        # Train on the SCALED inputs (X_train_scaled was created in Cell 1)
        model.fit(X_train_scaled, Y_train)
        
        # Predict on the unseen SCALED test data
        predictions = model.predict(X_test_scaled)
        
        # Calculate Uniform Metrics across all 16 TW targets
        r2 = r2_score(Y_test, predictions, multioutput='uniform_average')
        rmse = np.sqrt(mean_squared_error(Y_test, predictions))
        
        regression_results[name] = {'model': model, 'R2': r2, 'RMSE': rmse}
        
        # Print scoreboard row dynamically
        print(f"{name:<30} | {r2:>28.4f} | {rmse:>22.4f}")
        
    except Exception as e:
        print(f"{name:<30} | FAILED TO TRAIN: {str(e)[:40]}...")

🥊 THE 12-ALGORITHM MEGA BAKE-OFF 🥊
Algorithm Name                 | Avg R² Score (Higher=Better)   | Avg RMSE (Lower=Better)
------------------------------------------------------------------------------------------
Linear (Standard)              |                       0.7412 |                 2.8464
Linear (Ridge)                 |                       0.7391 |                 2.8590
Linear (Lasso)                 |                       0.4749 |                 2.8835
Linear (ElasticNet)            |                       0.5354 |                 4.1660
Distance (KNN)                 |                       0.7425 |                 8.9625
Distance (SVR)                 |                      -2.0341 |                29.7790
Tree (Decision Tree)           |                      -0.0228 |                 4.7020
Tree (Random Forest)           |                       0.6612 |                 3.4561
Boosting (AdaBoost)            |                       0.7471 |                 3.8526
B

In [5]:
import joblib
import os

# 1. Ensure the models directory exists in your project folder
os.makedirs('../models', exist_ok=True)

# 2. Extract the absolute best model from our Bake-Off dictionary
winner_name = "Boosting (Gradient Boost)"
best_model = regression_results[winner_name]['model']

# 3. Define the exact file paths for your FastAPI backend
scaler_path = '../models/tw_scaler.pkl'
model_path = '../models/tw_predictor.pkl'

# 4. Save the artifacts!
joblib.dump(scaler, scaler_path)
joblib.dump(best_model, model_path)

print("Done")
print("="*50)
print(f"🥇 Winning Algorithm: {winner_name}")
print(f"📊 Model Saved To:   {model_path}")
print(f"📏 Scaler Saved To:  {scaler_path}")

Done
🥇 Winning Algorithm: Boosting (Gradient Boost)
📊 Model Saved To:   ../models/tw_predictor.pkl
📏 Scaler Saved To:  ../models/tw_scaler.pkl
